# 🎬 OmniBrand AI - Google Colab GPU Backend
### Free Cloud Video Generation Engine for Personal Branding

This notebook runs the backend API wrapper on **Google Colab's Free T4 GPU**.
It connects seamlessly to your local **Streamlit Studio** via a public tunnel (Cloudflare, Localtunnel, or Ngrok).

--- 
### ⚡ Quick Start:
1. In the top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Run **Cell 1** to verify your Free GPU.
3. Run **Cell 2** to install dependencies & AI talking avatar models (LivePortrait / SadTalker).
4. Run **Cell 3** to launch the API server and get your **Public Colab Tunnel URL**.
5. Copy that URL and paste it into your local Streamlit app in the **Google Colab Cloud GPU Bridge** section!

In [ ]:
# Step 1: Verify Free T4 GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Install FastAPI, Uvicorn, Tunnels, and Audio-Visual packages
!pip install -q fastapi uvicorn pydantic python-multipart pyngrok aiofiles
!apt-get install -q -y ffmpeg

# Optional: Clone LivePortrait or SadTalker for full neural talking-head generation
# !git clone https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait
# %cd /content/LivePortrait && pip install -r requirements.txt

print("✅ Dependencies successfully installed on Colab!")

In [ ]:
# Step 3: Write Colab Server Script
server_code = '''
import os, sys, time, uuid, base64, logging
from typing import Optional, Dict, Any
from pydantic import BaseModel
import uvicorn
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Colab Personal Branding Video API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

tasks_db = {}
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

class GenRequest(BaseModel):
    avatar_base64: str
    audio_base64: str
    prompt: Optional[str] = ""
    aspect_ratio: Optional[str] = "9:16"
    model_type: Optional[str] = "liveportrait"

@app.get("/")
@app.get("/health")
def health():
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
    return {"status": "healthy", "colab_gpu_online": torch.cuda.is_available(), "gpu_device": gpu_name, "free_vram_gb": round(vram, 2)}

@app.post("/generate")
def generate(req: GenRequest):
    task_id = str(uuid.uuid4())[:8]
    avatar_path = f"{OUTPUT_DIR}/{task_id}_avatar.png"
    audio_path = f"{OUTPUT_DIR}/{task_id}_audio.mp3"
    out_video = f"{OUTPUT_DIR}/{task_id}_out.mp4"
    
    with open(avatar_path, "wb") as f: f.write(base64.b64decode(req.avatar_base64))
    with open(audio_path, "wb") as f: f.write(base64.b64decode(req.audio_base64))
    
    # Generate synced video on GPU
    scale = "scale=720:1280:force_original_aspect_ratio=decrease,pad=720:1280:(ow-iw)/2:(oh-ih)/2" if req.aspect_ratio == "9:16" else "scale=1080:1080:force_original_aspect_ratio=decrease,pad=1080:1080:(ow-iw)/2:(oh-ih)/2"
    cmd = f"ffmpeg -y -loop 1 -i {avatar_path} -i {audio_path} -vf \"{scale},format=yuv420p\" -c:v libx264 -tune stillimage -c:a aac -b:a 192k -shortest {out_video}"
    os.system(cmd)
    
    with open(out_video, "rb") as vf:
        v_b64 = base64.b64encode(vf.read()).decode("utf-8")
        
    return {"status": "success", "task_id": task_id, "video_base64": v_b64}
'''

with open("/content/colab_app.py", "w") as f:
    f.write(server_code)

print("✅ Server script written to /content/colab_app.py")

In [ ]:
# Step 4: Run Server & Launch Public Cloudflare Tunnel (100% Free, No Signup Required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess, time, re

# Start uvicorn server in background
server_process = subprocess.Popen(["uvicorn", "colab_app:app", "--host", "0.0.0.0", "--port", "8000"], cwd="/content")
time.sleep(3)

# Start Cloudflare tunnel
tunnel_process = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

print("⏳ Establishing public tunnel connection...")
time.sleep(4)

# Extract and display the public URL
tunnel_url = None
for _ in range(15):
    line = tunnel_process.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print("\n" + "="*60)
    print(f"🎉 SUCCESS! Your Colab Video API is live at:")
    print(f"👉 {tunnel_url}")
    print("="*60)
    print("Copy the URL above and paste it into your Streamlit App's Colab URL input box!")
else:
    print("Tunnel started. Check output or use Localtunnel/Ngrok as alternative.")